
# Vision RAG with BLIP-style Captioning + Groq + Gradio

## Features
- Image captioning (BLIP-style prompt via Groq Vision)
- ~10 word explanation
- Follow-up Q&A on the image
- Interactive professional Gradio UI
- Secure API key input


In [ ]:

!pip install -U groq gradio pillow


In [ ]:

from groq import Groq
from PIL import Image
import gradio as gr
import base64
import io


In [ ]:

def encode_image(image):
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


In [ ]:

def generate_caption(image, api_key):
    try:
        client = Groq(api_key=api_key)
        img_base64 = encode_image(image)

        response = client.chat.completions.create(
            model="llama-3.2-11b-vision-preview",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Describe this image in about 10 words only."},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                        },
                    ],
                }
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return str(e)


In [ ]:

def follow_up_question(image, question, api_key):
    try:
        client = Groq(api_key=api_key)
        img_base64 = encode_image(image)

        response = client.chat.completions.create(
            model="llama-3.2-11b-vision-preview",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": question},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                        },
                    ],
                }
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return str(e)


In [ ]:

def app():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🧠 Vision RAG Assistant (Groq + BLIP-style)")
        gr.Markdown("Upload an image → Get a concise explanation → Ask follow-up questions")

        with gr.Row():
            with gr.Column():
                api_key = gr.Textbox(label="Groq API Key", type="password")
                image = gr.Image(type="pil", label="Upload Image")

                caption_btn = gr.Button("Generate 10-word Explanation")

            with gr.Column():
                caption_output = gr.Textbox(label="Image Explanation")

        gr.Markdown("## Ask Follow-up Questions")

        question = gr.Textbox(label="Your Question")
        answer = gr.Textbox(label="Answer")

        ask_btn = gr.Button("Ask")

        caption_btn.click(generate_caption, inputs=[image, api_key], outputs=caption_output)
        ask_btn.click(follow_up_question, inputs=[image, question, api_key], outputs=answer)

    return demo

demo = app()
demo.launch()
